# Normalizing flows from zero: a practical RealNVP seminar

**Standalone Colab notebook · about 90 minutes · no external files or dataset downloads**

A normalizing flow learns a **reversible change of coordinates**:

$$
\text{simple Gaussian }z
\;\xleftrightarrow[\text{density evaluation}]{\text{generation}}\;
\text{complicated data }x.
$$

By the end, you should be able to:

1. explain why a change in volume changes probability density;
2. use the change-of-variables formula without confusing its direction;
3. explain why a RealNVP coupling layer is invertible and has a cheap Jacobian determinant;
4. train a small flow by maximum likelihood, evaluate densities, and generate samples;
5. distinguish RealNVP, MAF, IAF, Glow, spline flows, continuous normalizing flows, and flow matching.

**Suggested timing:** intuition 0–15, change of variables 15–30, coupling layers 30–50,
implementation and checks 50–65, training and diagnosis 65–80, modern context 80–90 minutes.

In [ ]:
# Colab already provides these packages. Locally, install torch, numpy, matplotlib and pandas.
import os, sys, time, math, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

START = time.perf_counter()
SEED = 20261105
SMOKE_TEST = os.environ.get('NF_SMOKE_TEST', '0') == '1'

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(2, os.cpu_count() or 1))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({
    'figure.dpi': 115,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

versions = {'python': sys.version.split()[0]}
for package in ['torch', 'numpy', 'matplotlib', 'pandas']:
    versions[package] = importlib.metadata.version(package)
print('Device:', device)
display(pd.Series(versions, name='version').to_frame())

## 1. The big idea · 0–15 min

Imagine a flexible sheet covered with dots. We stretch and bend the sheet until a simple round
cloud becomes the shape of our data. A normalizing flow learns that stretching operation.

It must satisfy two rules:

- **Invertible:** every output has exactly one input, so we can move in both directions.
- **Trackable volume change:** we can calculate how much every small region was stretched or squeezed.

We use two names for the same reversible map:

$$
x=g_\theta(z) \quad\text{(Gaussian to data: generation)},
\qquad
z=f_\theta(x)=g_\theta^{-1}(x) \quad\text{(data to Gaussian: density evaluation)}.
$$

The base distribution is usually

$$p_Z(z)=\mathcal N(0,I).$$

Unlike a VAE, the basic flow has no separate stochastic encoder and decoder: one map is the exact
inverse of the other. Unlike a GAN, it can assign a tractable density to an input.

In [ ]:
# Our dataset is generated here: no internet, file paths or authentication are needed.
def make_two_moons(n, noise=0.09, seed=SEED):
    rng = np.random.default_rng(seed)
    n1, n2 = n // 2, n - n // 2
    a = rng.uniform(0, np.pi, n1)
    b = rng.uniform(0, np.pi, n2)
    first = np.c_[np.cos(a), np.sin(a)]
    second = np.c_[1 - np.cos(b), 0.45 - np.sin(b)]
    points = np.vstack([first, second])
    points += rng.normal(0, noise, points.shape)
    rng.shuffle(points)
    return torch.tensor(points, dtype=torch.float32)

raw = make_two_moons(6000)
raw_train, raw_valid = raw[:5000], raw[5000:]
data_mean, data_std = raw_train.mean(0), raw_train.std(0)
x_train = (raw_train - data_mean) / data_std
x_valid = (raw_valid - data_mean) / data_std

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*x_train[:1800].T, s=7, alpha=.45)
ax.set(title='Training data: two moons', xlabel='$x_1$', ylabel='$x_2$', aspect='equal')
plt.show()
print('train:', tuple(x_train.shape), 'validation:', tuple(x_valid.shape))

**Notice:** a single Gaussian cannot describe these two curved bands. The flow will try to
“unbend” the moons into a Gaussian when moving from (x) to (z).

### First vocabulary check

- A **sample** is one point, such as ((x_1,x_2)).
- A **density** says how concentrated probability is around a point. It is not the probability
  of that exact continuous point; that probability is zero.
- **Likelihood** treats the observed data as fixed and asks which model parameters make it dense.
- **Negative log-likelihood (NLL)** is the loss we minimize.

## 2. Why stretching changes density · 15–25 min

Start in one dimension. Let

$$z\sim\mathcal N(0,1), \qquad x=az+b.$$

If (|a|=2), an interval of width 1 becomes width 2. The same probability is spread across twice
as much space, so the density becomes half as tall:

$$
p_X(x)=p_Z\!\left(\frac{x-b}{a}\right)\frac{1}{|a|}.
$$

The factor (1/|a|) is the volume correction. In many dimensions, the absolute determinant of
the Jacobian plays the role of (|a|).

In [ ]:
# Experiment: transform Gaussian samples with x = a*z + b.
a, b = 2.0, 1.0
z = torch.randn(30_000)
x = a * z + b
grid = torch.linspace(-6, 8, 500)
standard_normal = torch.distributions.Normal(0.0, 1.0)
transformed_density = standard_normal.log_prob((grid - b) / a).exp() / abs(a)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(x.numpy(), bins=80, density=True, alpha=.35, label='samples $x=2z+1$')
ax.plot(grid, transformed_density, lw=2, label='change-of-variables density')
ax.set(xlabel='$x$', ylabel='density', title='Stretch by 2 → density is half as tall')
ax.legend()
plt.show()

print('Approximate integral of the density:',
      float(torch.trapz(transformed_density, grid)))

**What to learn:** probability was not destroyed by stretching—the curve still integrates to
approximately 1. The density height changed because density means “probability per unit length.”

## 3. Change of variables and the Jacobian · 25–35 min

For a vector function (f:\mathbb R^d\rightarrow\mathbb R^d), its Jacobian contains all local
partial derivatives:

$$
J_f(x)=\frac{\partial f(x)}{\partial x}.
$$

The value (|\det J_f(x)|) is the local **volume multiplier**. In two dimensions it measures area
change; in three dimensions it measures volume change.

We will implement the **data-to-base** direction (z=f_\theta(x)). Then

$$
\boxed{\log p_X(x)=\log p_Z(f_\theta(x))
+\log\left|\det J_{f_\theta}(x)\right|.}
$$

If you instead write the generation direction (x=g_\theta(z)), the same statement becomes

$$
\log p_X(g_\theta(z))=\log p_Z(z)
-\log\left|\det J_{g_\theta}(z)\right|.
$$

The signs differ because the determinant of an inverse map is the reciprocal determinant.

In [ ]:
# A linear map turns a unit square into a parallelogram.
A = np.array([[1.5, 0.5], [0.2, 0.8]])
square = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]], dtype=float)
warped = square @ A.T
area_multiplier = abs(np.linalg.det(A))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].fill(*square.T, alpha=.35); axes[0].plot(*square.T)
axes[0].set(title='Before: area = 1', xlim=(-.2, 2.2), ylim=(-.2, 1.5), aspect='equal')
axes[1].fill(*warped.T, alpha=.35); axes[1].plot(*warped.T)
axes[1].set(title=f'After: area = |det A| = {area_multiplier:.2f}',
            xlim=(-.2, 2.2), ylim=(-.2, 1.5), aspect='equal')
plt.tight_layout(); plt.show()
print('A =\n', A)
print('|det(A)| =', area_multiplier)

**The computational problem:** for a general (d\times d) Jacobian, calculating a determinant
costs roughly (O(d^3)). Images can have thousands or millions of dimensions. A normalizing flow
therefore uses specially designed invertible transformations whose determinants are cheap.

## 4. The RealNVP affine coupling trick · 35–50 min

Split the coordinates into two parts. Keep part (a) unchanged and use it to transform part (b).
In the data-to-base direction:

$$
z_a=x_a,
\qquad
z_b=(x_b-t(x_a))\odot\exp[-s(x_a)].
$$

The inverse is available immediately:

$$
x_a=z_a,
\qquad
x_b=z_b\odot\exp[s(z_a)]+t(z_a).
$$

The neural networks (s) and (t) do **not** need to be invertible. During inversion, the
unchanged half gives us their inputs. This creates a triangular Jacobian, so

$$
\log\left|\det\frac{\partial z}{\partial x}\right|
=-\sum_j s_j(x_a).
$$

One layer leaves half the coordinates unchanged. We therefore alternate masks so every coordinate
eventually gets transformed.

Source: [RealNVP, Dinh et al.](https://arxiv.org/abs/1605.08803)

In [ ]:
class AffineCoupling(nn.Module):
    # One RealNVP layer. forward: data x -> base z; inverse: z -> x.
    def __init__(self, dim, mask, hidden=64, scale_limit=1.5):
        super().__init__()
        self.register_buffer('mask', torch.tensor(mask, dtype=torch.float32))
        self.scale_limit = scale_limit

        def make_net():
            net = nn.Sequential(
                nn.Linear(dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, dim),
            )
            # Begin close to the identity map for stable training.
            nn.init.zeros_(net[-1].weight)
            nn.init.zeros_(net[-1].bias)
            return net

        self.scale_net = make_net()
        self.shift_net = make_net()

    def _scale_and_shift(self, conditioned):
        change = 1.0 - self.mask
        # tanh prevents exp(s) from becoming numerically enormous.
        s = self.scale_limit * torch.tanh(self.scale_net(conditioned)) * change
        t = self.shift_net(conditioned) * change
        return s, t

    def forward(self, x):
        # Data -> base; return transformed points and log|det dz/dx|.
        fixed = x * self.mask
        s, t = self._scale_and_shift(fixed)
        z = fixed + (1.0 - self.mask) * (x - t) * torch.exp(-s)
        log_abs_det = -s.sum(dim=1)
        return z, log_abs_det

    def inverse(self, z):
        # Base -> data.
        fixed = z * self.mask
        s, t = self._scale_and_shift(fixed)
        x = fixed + (1.0 - self.mask) * (z * torch.exp(s) + t)
        return x


class RealNVP(nn.Module):
    def __init__(self, dim=2, n_layers=8, hidden=64):
        super().__init__()
        masks = [[1, 0] if i % 2 == 0 else [0, 1] for i in range(n_layers)]
        self.layers = nn.ModuleList(
            [AffineCoupling(dim, mask, hidden) for mask in masks]
        )
        self.dim = dim

    def to_latent(self, x):
        log_abs_det = torch.zeros(x.shape[0], device=x.device)
        z = x
        for layer in self.layers:
            z, layer_log_det = layer(z)
            log_abs_det += layer_log_det
        return z, log_abs_det

    def from_latent(self, z):
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x

    @staticmethod
    def base_log_prob(z):
        return -0.5 * (z.square() + math.log(2 * math.pi)).sum(dim=1)

    def log_prob(self, x):
        z, log_abs_det = self.to_latent(x)
        return self.base_log_prob(z) + log_abs_det

    @torch.no_grad()
    def sample(self, n, device):
        z = torch.randn(n, self.dim, device=device)
        return self.from_latent(z)


model = RealNVP().to(device)
print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters()):,}')

### Why test invertibility and the determinant?

These are not decorative details: they are the mathematical contract of a flow. A sign error in
the log-determinant can still produce a decreasing loss while training the wrong density.

In [ ]:
# Check 1: x -> z -> x should recover the original point.
test_x = x_train[:16].to(device)
test_z, analytic_log_det = model.to_latent(test_x)
recovered_x = model.from_latent(test_z)
round_trip_error = (recovered_x - test_x).abs().max().item()

# Check 2: compare our cheap formula with PyTorch's full 2x2 Jacobian.
point = test_x[0].detach().clone().requires_grad_(True)
jacobian = torch.autograd.functional.jacobian(
    lambda v: model.to_latent(v.unsqueeze(0))[0].squeeze(0), point
)
_, numerical_log_det = torch.linalg.slogdet(jacobian)

print(f'Max round-trip error: {round_trip_error:.3e}')
print(f'Analytic log|det J|: {analytic_log_det[0].item(): .6f}')
print(f'Full-Jacobian value: {numerical_log_det.item(): .6f}')
assert round_trip_error < 1e-5
assert torch.allclose(analytic_log_det[0], numerical_log_det, atol=1e-5)

**What to learn:** we calculate the full Jacobian only as a tiny two-dimensional unit test.
Training uses the cheap coupling-layer formula. For high-dimensional data, the full Jacobian would
defeat the purpose of the architecture.

**Predict before training:**

1. Why will one fixed mask be insufficient?
2. Why use `tanh` on the predicted scale?
3. If the transformation expands local volume in the generation direction, should the resulting
   density increase or decrease?

## 5. Maximum-likelihood training · 50–70 min

For each data point, the model computes

$$
\log p_X(x)=
\underbrace{\log p_Z(f_\theta(x))}_{\text{put data in a likely Gaussian region}}
+
\underbrace{\log|\det J_{f_\theta}(x)|}_{\text{correct for local volume change}}.
$$

We minimize the mean negative log-likelihood:

$$
\mathcal L_{\mathrm{NLL}}
=-\frac{1}{N}\sum_{i=1}^N\log p_X(x_i).
$$

There is no discriminator and no ELBO: the density is evaluated directly for this flow.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_steps = 80 if SMOKE_TEST else 1200
batch_size = 512
report_every = 20 if SMOKE_TEST else 50
history = []

model.train()
for step in range(1, n_steps + 1):
    indices = torch.randint(len(x_train), (batch_size,))
    batch = x_train[indices].to(device)

    loss = -model.log_prob(batch).mean()
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    optimizer.step()

    if step == 1 or step % report_every == 0:
        model.eval()
        with torch.no_grad():
            valid_nll = -model.log_prob(x_valid.to(device)).mean().item()
        history.append((step, loss.item(), valid_nll))
        model.train()

history = pd.DataFrame(history, columns=['step', 'train NLL', 'validation NLL'])
display(history.tail())

In [ ]:
ax = history.plot(x='step', y=['train NLL', 'validation NLL'], figsize=(7, 3.5))
ax.set(title='Maximum-likelihood training', ylabel='negative log-likelihood')
ax.grid(alpha=.2)
plt.show()

# Repeat the mathematical contract tests after learning a non-trivial transform.
model.eval()
point = x_valid[0].to(device).detach().clone().requires_grad_(True)
z_one, analytic = model.to_latent(point.unsqueeze(0))
jacobian = torch.autograd.functional.jacobian(
    lambda v: model.to_latent(v.unsqueeze(0))[0].squeeze(0), point
)
_, numerical = torch.linalg.slogdet(jacobian)
reconstructed = model.from_latent(z_one)
print('Post-training round-trip max error:',
      float((reconstructed - point.unsqueeze(0)).abs().max()))
print('Post-training analytic vs full log-det:',
      round(analytic.item(), 5), round(numerical.item(), 5))
assert torch.allclose(analytic.squeeze(), numerical, atol=2e-4)

**What to learn:** training and validation NLL should generally fall together. Mini-batch noise is
normal. A lower training NLL with a rising validation NLL would indicate overfitting. The numerical
checks confirm that learning did not break invertibility or the determinant calculation.

## 6. Does the learned model do both jobs? · 70–80 min

A trained flow should support:

- **density evaluation:** map (x\to z), calculate the base density and volume correction;
- **generation:** sample (z\sim\mathcal N(0,I)), then apply the exact inverse (z\to x).

In [ ]:
model.eval()
with torch.no_grad():
    generated = model.sample(3000, device).cpu()
    real_latent, _ = model.to_latent(x_valid.to(device))
    real_latent = real_latent.cpu()
    base_samples = torch.randn_like(real_latent)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].scatter(*x_valid.T, s=8, alpha=.45)
axes[0].set(title='Held-out real data', xlabel='$x_1$', ylabel='$x_2$', aspect='equal')
axes[1].scatter(*generated.T, s=8, alpha=.35, color='tab:orange')
axes[1].set(title='Generated samples', xlabel='$x_1$', ylabel='$x_2$', aspect='equal')
axes[2].scatter(*real_latent.T, s=8, alpha=.35, label='$f(x)$')
axes[2].scatter(*base_samples.T, s=8, alpha=.2, label='$N(0,I)$')
axes[2].set(title='Data mapped to latent space', xlabel='$z_1$', ylabel='$z_2$', aspect='equal')
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Visualize the learned density. This is practical in 2D; image-space grids are impossible.
lo = torch.minimum(x_train.min(0).values, generated.min(0).values) - .4
hi = torch.maximum(x_train.max(0).values, generated.max(0).values) + .4
gx = torch.linspace(lo[0], hi[0], 180)
gy = torch.linspace(lo[1], hi[1], 180)
xx, yy = torch.meshgrid(gx, gy, indexing='xy')
grid = torch.stack([xx.ravel(), yy.ravel()], dim=1)
with torch.no_grad():
    density = model.log_prob(grid.to(device)).exp().cpu().reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
# Convert the 2D tensors explicitly for compatibility across Matplotlib versions.
contour = ax.contourf(xx.numpy(), yy.numpy(), density.numpy(), levels=30, cmap='Blues')
ax.scatter(*x_valid[:500].T, s=7, c='tab:orange', alpha=.45, label='held-out data')
ax.set(title='Learned probability density', xlabel='$x_1$', ylabel='$x_2$', aspect='equal')
ax.legend(fontsize=8)
fig.colorbar(contour, ax=ax, label='$p_\theta(x)$')
plt.show()

**What to learn:** successful generation and successful density estimation are two directions of
the same learned bijection. Expect imperfections: an invertible continuous map from a Gaussian has
full connected support, so it cannot create two exactly disconnected zero-density islands. It can
approximate them using a very thin, low-density bridge.

## 7. Which flow is which? · 80–85 min

| Method | Main idea | Density evaluation | Sampling |
|---|---|---:|---:|
| NICE | additive coupling | fast | fast |
| RealNVP | affine coupling | fast | fast |
| MAF | autoregressive map arranged for (x\to z) | fast/parallel | sequential |
| IAF | inverse autoregressive direction | sequential | fast/parallel |
| Glow | ActNorm + invertible (1\times1) convolution + coupling | fast | fast |
| Neural spline flow | replace affine transforms with monotone splines | fast | fast |
| Continuous NF / FFJORD | integrate a neural ODE; track density through divergence | ODE solve | ODE solve |

“Forward” and “inverse” names are easy to misuse. Always ask the operational question:
**Is this direction data-to-base or base-to-data, and is it parallel or sequential?**

Primary sources: [MAF](https://papers.neurips.cc/paper/6828-masked-autoregressive-flow-for-density-estimation),
[Glow](https://proceedings.neurips.cc/paper/2018/hash/d139db6a236200b21cc7f752979132d0-Abstract.html),
[Neural Spline Flows](https://proceedings.neurips.cc/paper_files/paper/2019/hash/7ac71d433f282034e088473244df8c02-Abstract.html),
[FFJORD](https://arxiv.org/abs/1810.01367).

## 8. Practical image details and modern context · 85–90 min

### Discrete pixels need care

Pixel values such as 0, 1, ..., 255 are discrete, while the change-of-variables formula above is
for continuous densities. Image flows commonly add small uniform noise (**dequantization**) before
modeling and may apply a logit-like transform. Otherwise a continuous model can concentrate density
around discrete points in pathological ways.

### Exact likelihood is useful, but it is not the same as visual quality

Likelihood, perceptual sample quality, and out-of-distribution detection are different questions.
Deep density models can sometimes assign unexpectedly high likelihood to out-of-distribution data.
Do not use likelihood as the only safety or quality metric.

Source: [Nalisnick et al., “Do Deep Generative Models Know What They Don't Know?”](https://arxiv.org/abs/1810.09136)

### Where flow matching fits

Modern **flow matching** trains a time-dependent velocity field that transports noise to data along
a continuous probability path. It is built on continuous normalizing flows, but its common training
objective regresses a vector field instead of calculating a discrete RealNVP log-determinant at each
layer. Therefore, “normalizing flow” and “flow matching” are related, not interchangeable terms.

Source: [Lipman et al., Flow Matching for Generative Modeling, ICLR 2023](https://arxiv.org/abs/2210.02747)

## Final check: what students should leave with

Complete these sentences without looking back:

1. A normalizing flow turns a ______ distribution into a ______ distribution using a ______ map.
2. The Jacobian determinant measures local ______ change.
3. We add the log-determinant in the (x\to z) formula because ______.
4. A coupling layer can use an ordinary non-invertible neural network for (s,t) because ______.
5. Alternating masks matters because ______.
6. Training minimizes ______, while generation starts by sampling ______.
7. Exact likelihood does not automatically guarantee ______.

**Core answer in one sentence:** a normalizing flow learns a reversible map between simple noise and
data, and uses the Jacobian determinant to keep density mathematically correct while space is warped.

In [ ]:
print(f'Notebook finished in {time.perf_counter() - START:.1f} seconds on {device}.')
print('All core experiments completed successfully.')